# leodragkit aero tutorial: building and plotting a spacecraft

A running example that grows with each aero task: every task adds a section showing what its code produces. Tests check the numbers; this notebook shows the result.

**Before running.** Activate the `leodragkit` environment and install the optional plotting library: `conda install -c conda-forge plotly`. The figures are interactive: drag to rotate, scroll to zoom, click a legend entry to hide it.

**In git.** Clear all outputs before committing (in Jupyter or VS Code: *Clear All Outputs*), so the repository holds code and text, not large figures.

| Section | Task | What it shows |
| --- | --- | --- |
| 1 | A3a | one triangle: corner order sets the outward normal |
| 2 | A3a | a cube built by hand as a `PanelSet` |
| 3 | A3p | the cube in 3-D, coloured by component and by material |
| 4 | A3a, A3p | a macro model: panels without positions |
| 5 | A3b1 | primitives (box, plate, cylinder) at two resolutions, placed by poses |
| 6 | A3b2 | a 3U CubeSat joined from components |


In [ ]:
# %load_ext autoreload
# %autoreload 2

import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
import numpy as np

from leodragkit.aero.geometry import (
    compute_triangle_properties,
    panel_set_from_panels,
    panel_set_from_triangles,
)
from leodragkit.plots.spacecraft import plot_panel_set_3d

## 1. One triangle (A3a)

A triangle's outward normal follows its corner order: counter-clockwise when seen from outside. These corners go counter-clockwise seen from +z, so the normal points along +z. The area is half the length of the cross product of two edges.

In [ ]:
triangle_m = np.array([[[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]])
properties = compute_triangle_properties(triangle_m)
print("normal:  ", properties.outward_normals[0])
print("area:    ", properties.areas_m2[0], "m²")
print("centroid:", properties.centroids_m[0], "m")

Listing the same corners in the opposite order flips the normal: the panel now faces −z.

In [ ]:
reversed_properties = compute_triangle_properties(triangle_m[:, ::-1, :])
print("normal with the corners reversed:", reversed_properties.outward_normals[0])

## 2. A cube built by hand (A3a)

A unit cube centred on the body origin: six faces, two triangles each, every face counter-clockwise seen from outside. Task A3b will replace this by-hand construction with primitives (`box`, `plate`, `cylinder`); writing it out once shows what those primitives produce.

The four side faces are the component `body` and the two z faces are `end_caps`; the ±y faces are `solar_cell` material and the rest `aluminium`.

In [ ]:
corners_m = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)])
faces = [
    (0, 1, 3, 2),
    (4, 6, 7, 5),
    (0, 4, 5, 1),
    (2, 3, 7, 6),
    (0, 2, 6, 4),
    (1, 5, 7, 3),
]  # −x +x −y +y −z +z
cube_triangles_m = np.array(
    [corners_m[list(triangle)] for a, b, c, d in faces for triangle in ((a, b, c), (a, c, d))]
)

cube = panel_set_from_triangles(
    cube_triangles_m,
    component_ids=[0] * 8 + [1] * 4,
    material_ids=[0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
    component_names=("body", "end_caps"),
    material_names=("aluminium", "solar_cell"),
    source="unit cube, built by hand",
)
print(cube.panel_count, "panels; total area", cube.areas_m2.sum(), "m²")
print("has positions:", cube.has_positions, "| has triangles:", cube.has_triangles)

## 3. The cube in 3-D (A3p)

Each component is one trace, so the legend lists them. The lines are the outward normals, one per triangle, starting at its centroid; the red, green and blue lines are the body x, y and z axes. Every normal should point away from the cube: one pointing inward would mean a triangle with its corners in the wrong order.

In [ ]:
figure = plot_panel_set_3d(cube)
figure

The same cube coloured by material: the ±y faces are solar cells.

In [ ]:
plot_panel_set_3d(cube, color_by="material")

## 4. A macro model: panels without positions (A3a, A3p)

Published macro models list each panel's outward normal, area and material, often with no position. Here is a simple one for a 3U CubeSat (0.1 × 0.1 × 0.34 m) with one deployed solar panel (0.3 × 0.1 m), listed once for each side. The normals are rounded to four decimals, as tables often are; `panel_set_from_panels` normalises them.

With no positions there is no shape to draw. The plot shows each normal from the origin, with length proportional to the panel's area, and the title says so. A set like this supports forces, but not shadowing or moments.

In [ ]:
normals = [
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],  # bus faces
    [0.7071, 0.7071, 0],
    [-0.7071, -0.7071, 0],  # one deployed panel, front and back, rounded
]
areas_m2 = [0.034, 0.034, 0.034, 0.034, 0.01, 0.01, 0.03, 0.03]
macro_model = panel_set_from_panels(
    normals,
    areas_m2,
    component_ids=[0] * 6 + [1] * 2,
    material_ids=[0] * 6 + [1] * 2,
    component_names=("bus", "deployed_panel"),
    material_names=("aluminium", "solar_cell"),
    source="3U CubeSat macro model (illustrative)",
)
plot_panel_set_3d(macro_model)

## 5. Primitives and poses (A3b1)

Plates, boxes and cylinders are built centred on their own origin, then placed in the body frame by a pose: a centre plus the component's own x and z axes (y = z × x, so the axes are always right-handed). `max_edge_length_m` sets the resolution; `None` gives the coarsest mesh. The triangle edges are drawn so the resolution is visible.

Until joining arrives in A3b2, a small helper turns one primitive's triangles into a `PanelSet` with a single component and material.

In [ ]:
from leodragkit.aero.geometry_builder import (
    build_box_triangles,
    build_cylinder_triangles,
    build_plate_triangles,
    make_pose,
    place_triangles,
)

In [ ]:
def single_component(triangles_m, name, material="aluminium"):
    """A PanelSet of one component and one material, for looking at a single primitive."""
    count = len(triangles_m)
    return panel_set_from_triangles(triangles_m, [0] * count, [0] * count, (name,), (material,), source=name)

A 3U CubeSat bus (0.1 × 0.1 × 0.34 m) at the coarsest resolution, then with no triangle edge longer than 2 cm. The force on the box is the same either way; the finer mesh only matters for shadowing.

In [ ]:
bus_coarse = single_component(build_box_triangles(0.1, 0.1, 0.34), "bus, coarsest")
bus_fine = single_component(build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=0.02), "bus, 2 cm")
print(bus_coarse.panel_count, "triangles coarse;", bus_fine.panel_count, "triangles at 2 cm")
plot_panel_set_3d(bus_fine)

In [ ]:
import sys

import numpy
import plotly

print(sys.executable, numpy.__version__, plotly.__version__)

A deployed solar panel (0.3 × 0.1 m, both sides), placed beside the bus and tilted by 45° about the body x axis: its own x stays along body x, and its normal (own z) turns from +z towards +y. The axes are typed rounded, as they would be in a YAML file; `make_pose` makes them exact.

In [ ]:
panel_pose = make_pose(centre_m=[0.0, 0.2, 0.0], x_axis=[1, 0, 0], z_axis=[0, 0.7071, 0.7071])
panel_m = place_triangles(build_plate_triangles(0.3, 0.1, max_edge_length_m=0.05, two_sided=True), panel_pose)
plot_panel_set_3d(single_component(panel_m, "deployed panel", "solar_cell"), color_by="material")

A capped cylinder, such as a boom or an antenna mast: the side is faceted into segments (at least eight), and the caps are fans from their centres. Every normal should point away from the axis or along it.

In [ ]:
boom_m = build_cylinder_triangles(0.02, 0.3, max_edge_length_m=0.02)
plot_panel_set_3d(single_component(boom_m, "boom"))

## 6. A spacecraft from components (A3b2)

A `Component` is one named part with one material, already placed in the body frame. `join_components` turns a list of them into one `PanelSet`: every triangle keeps its component and material, components of the same material share a material id, and the result is validated like any other `PanelSet`.

Here is a 3U CubeSat: an aluminium bus, long axis along z, with two solar panels deployed sideways from its top end, both facing +x and exposed on both sides, and a thin antenna along −z.

In [ ]:
from leodragkit.aero.geometry_builder import Component, join_components

resolution_m = 0.02
bus = Component("bus", "aluminium", build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=resolution_m))


def deployed_panel(name, side):
    """A 0.3 × 0.1 m two-sided panel beside the bus on the ±y side (side = +1 or −1), facing +x."""
    pose = make_pose(centre_m=[0.0, side * (0.05 + 0.15), 0.12], x_axis=[0, side, 0], z_axis=[1, 0, 0])
    plate = build_plate_triangles(0.3, 0.1, max_edge_length_m=resolution_m, two_sided=True)
    return Component(name, "solar_cell", place_triangles(plate, pose))


antenna_pose = make_pose(centre_m=[0.0, 0.0, -0.27], x_axis=[1, 0, 0], z_axis=[0, 0, 1])
antenna = Component(
    "antenna", "aluminium", place_triangles(build_cylinder_triangles(0.005, 0.2, capped=False), antenna_pose)
)

cubesat = join_components(
    [bus, deployed_panel("panel_plus_y", +1), deployed_panel("panel_minus_y", -1), antenna],
    source="3U CubeSat with two deployed panels (tutorial)",
)
print(cubesat.panel_count, "triangles")
print("components:", cubesat.component_names)
print("materials: ", cubesat.material_names)

Coloured by component, each part can be hidden from the legend; coloured by material, the two panels share one colour. The panels face +x on one side and −x on the other: as two-sided plates, both faces are exposed.

In [ ]:
plot_panel_set_3d(cubesat)

In [ ]:
plot_panel_set_3d(cubesat, color_by="material", show_edges=False)

## Saving a figure

Plot functions never write files. To keep a figure, save it yourself, for example into the output directory from your main config:

```python
figure.write_html("cube.html")
```

## Coming next

- **A3b3**: orientation checks and a geometry summary.
- **A3c**: the same spacecraft read from a YAML description.
- **A3d**: macro-model tables and STL or OBJ meshes from files.
- **A4**: the incoming flow drawn on the spacecraft (plot level 2).